Written by Lucie Reymondet, UCSD Scripps Institution of Oceanography

# Downloading products from EUMETSAT using the Data Access Catalog EUMDAC

Use EUMDAC's Python library to download from EUMETSAT Data Store
User Guide : 
https://user.eumetsat.int/resources/user-guides/eumetsat-data-access-client-eumdac-guide

## Initialize token

In [4]:
import datetime
import eumdac
import shutil
import time
import fnmatch
import requests
from IPython.core.display import HTML
import time
import os
import zipfile
import json
import xarray as xr

# Insert personal key and secret
consumer_key = '2WA_GJftbSnmegkQ6gNBxSqo_Foa'
consumer_secret = 'khEoQY_BjsQD1QDZ83ntHE9gIBoa'

# Generate token, by default validity = 24h
credentials = (consumer_key, consumer_secret)
token = eumdac.AccessToken(credentials)
try:
    print(f"This token '{token}' expires {token.expiration}")
except requests.exceptions.HTTPError as error:
    print(f"Error when tryng the request to the server: '{error}'")

# Initialize DataStore and DataTailor instances
datastore = eumdac.DataStore(token)
datatailor = eumdac.DataTailor(token)

This token '91e5d562-f6e9-3b2c-aac0-817bfe47f030' expires 2025-12-13 14:00:53.145399


## Check available products inside a collection

In [2]:
# Desired collection
coll = 'EO:EUM:DAT:0662' # MTG FCI NR

# Display search options for the selected collection
try:
    selected_collection = datastore.get_collection(coll) 
    print(f"{selected_collection} - {selected_collection.title}")
    print(f"Description: {selected_collection.abstract}")
    print(f"Metadata: {selected_collection.metadata}")
    print(f"Search options: {selected_collection.search_options} \n")
except eumdac.datastore.DataStoreError as error:
    print(f"Error related to the data store: '{error}'")
except eumdac.collection.CollectionError as error:
    print(f"Error related to the collection: '{error}'")
except requests.exceptions.ConnectionError as error:
    print(f"Error related to the connection: '{error}'")
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: {error}")

# Filter criteria
start = datetime.datetime(2025, 1, 1, 0, 0)
end = datetime.datetime(2025, 1, 1, 6, 0)

# Retrieve and display products that match our filter
products = selected_collection.search(
    dtstart=start, dtend=end
    )
try:
    print(f'{products.total_results} products found for the given filters:')
    for product in products:
        print(product)
except eumdac.collection.CollectionError as error:
    print(f"Error related to the collection: '{error}'")
except requests.exceptions.ConnectionError as error:
    print(f"Error related to the connection: '{error}'")
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: {error}")

EO:EUM:DAT:0662 - FCI Level 1c Normal Resolution Image Data - MTG - 0 degree
Description: The rectified (Level 1c) Meteosat FCI full disc image data in normal spatial (FDHSI) resolution. The FCI instrument consists of 16 imaging spectral channels ranging from 0.4 µm to 13.3 µm with the channel at 3.8 µm having an extended dynamic range dedicated to fire monitoring. The spatial resolution is 1km for visible and near-infrared channels and 2 km for infrared channels. FCI Level 1c rectified radiance dataset consists of a set of files that contain the level 1c science data rectified to a reference grid together with the auxiliary data associated with the processing configuration and the quality assessment of the dataset.
Metadata: {'geometry': {'type': 'Polygon', 'coordinates': [[[-79, 79], [-79, -79], [79, -79], [79, 79], [-79, 79]]]}, 'properties': {'type': 'Properties', 'kind': 'Dataset', 'identifier': 'EO:EUM:DAT:0662', 'title': 'FCI Level 1c Normal Resolution Image Data - MTG - 0 degre

## Download products

Products can be downloaded by providing either their product ID, or a combination of their collection ID and several search parameters. We can download all products resulting from a search, or a single product. We can also download entire products or specific file components (e.g. metadata only), or chunks based on geographical coverage.

In [5]:
# Option 1 : Download all products resulting from a search
t0 = time.time()

start = datetime.datetime(2025, 1, 1, 0, 0)
end = datetime.datetime(2025, 1, 1, 6, 0)
products = selected_collection.search(
    dtstart=start, dtend=end
    )

dirout = r"C:\Users\lureymondet\GitHub\GOFLOW_LR\data\raw"

for product in products:
    try:
        with product.open() as fsrc:
            target = os.path.join(dirout,os.path.basename(fsrc.name)) 
            with open(target, mode='wb') as fdst:
                shutil.copyfileobj(fsrc, fdst)
            print(f'Download of product {product} finished.')
    except eumdac.product.ProductError as error:
        print(f"Error related to the product '{product}' while trying to download it: '{error.msg}'")
    except requests.exceptions.ConnectionError as error:
        print(f"Error related to the connection: '{error}'")
    except requests.exceptions.RequestException as error:
        print(f"Unexpected error: {error}")
    
print(f'All downloads are finished. Time elapsed : {(time.time()-t0)/60:.1f} min')

Download of product W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20250101055303_IDPFI_OPE_20250101055007_20250101055924_N__O_0036_0000 finished.
Download of product W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20250101054257_IDPFI_OPE_20250101054007_20250101054924_N__O_0035_0000 finished.
Download of product W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20250101053256_IDPFI_OPE_20250101053007_20250101053924_N__O_0034_0000 finished.
Download of product W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20250101052258_IDPFI_OPE_20250101052007_20250101052924_N__O_0033_0000 finished.
Download of product W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20250101051257_IDPFI_OPE_20250101051007_20250101051924_N__O_0032_0000 finished.


ProtocolError: ('Connection broken: IncompleteRead(331128287 bytes read, 374838979 more expected)', IncompleteRead(331128287 bytes read, 374838979 more expected))